# Neo4j Graph RAG Notebook (GPT version)

## 1. Install dependencies

In [14]:
import os
from dotenv import load_dotenv, dotenv_values 
load_dotenv() 


True

## 2. Environment variables

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.chains import GraphCypherQAChain
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_community.graphs import Neo4jGraph
from langchain_openai import ChatOpenAI


## 3. Load PDFs from folder

In [16]:
loader = PyPDFDirectoryLoader("data/")
docs = loader.load()

print(f"Loaded {len(docs)} pages")

Loaded 155 pages


## 4. Split documents

In [17]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

documents = splitter.split_documents(docs)
print(f"Created {len(documents)} chunks")

Created 1237 chunks


## 5. Neo4j connection

In [18]:
graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"),
    username=os.getenv("NEO4J_USERNAME"),
    password=os.getenv("NEO4J_PASSWORD"),
)

## 6. GPT LLM setup 

In [19]:
llm = ChatOpenAI(
    api_key=os.getenv("OpenAi_api"),
    base_url="https://aigateway.ntictsolution.com/v1",
    model="gpt-4o",
    temperature=0
)

## 7. Graph Transformer

In [20]:
graph_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=[
        "Entity",
        "Concept",
        "Process",
        "Document"
    ],
    allowed_relationships=[
        "MENTIONS",
        "DESCRIBES",
        "RELATED_TO",
        "PART_OF",
        "USES"
    ],
    node_properties=["name", "source"]
)


## 8. Build graph

In [21]:
import time
from tqdm import tqdm

# Check if graph already exists
result = graph.query("MATCH (n) RETURN count(n) AS c")
count = result[0]["c"] if result else 0

if count == 0:
    print("Building knowledge graph...")

    start_all = time.time()

    for i, doc in enumerate(tqdm(documents, desc="Building graph chunks")):
        start = time.time()

        try:
            tqdm.write(f"➡️ Sending chunk {i+1} to GPT...")
            gdocs = graph_transformer.convert_to_graph_documents([doc])
            tqdm.write(f"⬅️ GPT responded for chunk {i+1}")

            graph.add_graph_documents(
                gdocs,
                include_source=True
            )


            elapsed = time.time() - start
            tqdm.write(f"Chunk {i+1}/{len(documents)} done in {elapsed:.2f}s")

        except Exception as e:
            tqdm.write(f"!Chunk {i+1} failed: {e}")

    total = time.time() - start_all
    print(f"\nGraph created successfully in {total/60:.2f} minutes")

else:
    print("Graph already exists — skipping build")


Graph already exists — skipping build


## 9. Graph RAG QA

In [22]:
qa_chain = GraphCypherQAChain.from_llm(
llm=llm,
graph=graph,
verbose=True
)

In [23]:
qa_chain.run("What happpen in 2022?")



> Entering new GraphCypherQAChain chain...
Generated Cypher:
cypher
MATCH (d:Document)-[:MENTIONS|DESCRIBES|RELATED_TO]->(e:Entity)
WHERE d.text CONTAINS "2022"
RETURN d, e

Full Context:
[{'d': {'id': '6e629cf8a2f0241a972b9ad3af13d681', 'source': 'data\\2023_Formula_One_World_Championship.pdf', 'text': "Do NOT infer, guess, or assume.\nIf unsure, skip.\nSebastian Vettel retired at the end of the 2022 championship,[34] ending his Formula One career after 15 full seasons.[35] His place at Aston Martin wastaken by Fernando Alonso, who left Alpine after two seasons.[36] Alonso's replacement was initially announced as the 2021 Formula 2 Champion andAlpine reserve driver, Oscar Piastri.[37] Shortly after this announcement, Piastri stated that he had not signed a contract for 2023 and that he would not\nFree practice drivers\nTeam changes\nDriver changes", 'page': 1}, 'e': {'name': 'Fernando Alonso', 'id': 'Fernando Alonso'}}, {'d': {'id': '6e629cf8a2f0241a972b9ad3af13d681', 'source': 'dat

'Sebastian Vettel retired at the end of the 2022 championship, ending his Formula One career after 15 full seasons. Daniel Ricciardo left McLaren after two seasons, with his contract for 2023 being terminated by mutual agreement during the 2022 championship. Nicholas Latifi left Williams after three seasons, and Mick Schumacher left Haas after two seasons.'

## SUMMARY

not that good solo, good in explaining but not that good in searching not spasific name( maybe causing from graph building using GPT need to try neo4j aura method)